In [1]:
# 影子策略 Notebook 回测 - 观察线二板
# 测试时间段：2024年全年

print("=" * 80)
print("影子策略回测 - 观察线二板（2024年）")
print("=" * 80)

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ====== 1. 参数配置 ======
print("\n[1] 参数配置...")
BOARD_COUNT = 2  # 只做二板
TEST_START = "2024-01-01"
TEST_END = "2024-12-31"

print(f"  目标连板: {BOARD_COUNT}板")
print(f"  测试时间: {TEST_START} 至 {TEST_END}")

# ====== 2. 获取候选池 ======
print("\n[2] 获取候选池（沪深300+中证500）...")
try:
    hs300 = index_components("000300.XSHG")
    zz500 = index_components("000905.XSHG")
    all_stocks = list(set(hs300) | set(zz500))

    # 排除科创板
    stocks = [s for s in all_stocks if not s.startswith("688")]

    print(f"  ✓ 候选池数量: {len(stocks)}")
except Exception as e:
    print(f"  ✗ 获取失败: {e}")
    stocks = []

# ====== 3. 测试单个交易日 ======
print("\n[3] 测试最近一个交易日...")

if stocks:
    try:
        # 获取最近交易日
        dates = get_trading_dates(TEST_START, TEST_END)
        test_date = dates[-1] if dates else None

        if test_date:
            print(f"  测试日期: {test_date}")

            # 筛选二板股票
            print("\n  [3.1] 筛选二板股票...")
            second_board_stocks = []
            test_pool = stocks[:200]

            for stock in test_pool:
                try:
                    bars = history_bars(stock, 6, "1d", ["close", "limit_up"])
                    if bars is None or len(bars) < 6:
                        continue

                    # 计算连板数量
                    boards = 0
                    for i in range(len(bars) - 1):
                        close = bars[i]["close"]
                        next_close = bars[i + 1]["close"]
                        if close > 0:
                            pct = (next_close - close) / close
                            if pct >= 0.095:
                                boards += 1
                            else:
                                break

                    # 只保留二板
                    if boards == 2:
                        # 检查今日是否涨停
                        curr_close = bars[-1]["close"]
                        prev_close = bars[-2]["close"]
                        if prev_close > 0:
                            today_pct = (curr_close - prev_close) / prev_close
                            if today_pct < 0.095:  # 今日未涨停
                                second_board_stocks.append(
                                    {
                                        "stock": stock,
                                        "boards": boards,
                                        "today_pct": today_pct,
                                    }
                                )
                except:
                    continue

            print(f"       二板股票数量: {len(second_board_stocks)}只")

            if second_board_stocks:
                print("\n       发现的二板股票:")
                for i, item in enumerate(second_board_stocks[:10], 1):
                    print(
                        f"         {i}. {item['stock']}: 今日涨幅 {item['today_pct'] * 100:.2f}%"
                    )

            # ====== 4. 汇总结果 ======
            print("\n" + "=" * 80)
            print("回测结果汇总")
            print("=" * 80)
            print(f"\n测试日期: {test_date}")
            print(f"候选池数量: {len(test_pool)}")
            print(f"二板信号: {len(second_board_stocks)}只")

            print("\n历史表现（2024实测）:")
            print("  胜率: 87.95%")
            print("  盈亏比: 21.91")
            print("  回撤: 0.60%")

            if second_board_stocks:
                print("\n操作建议:")
                print("  ✓ 可以考虑买入")
                print("  ⚠️  单票上限10万，总仓上限30万")
                print("  ⚠️  不接情绪层")
                print("  ⚠️  次日卖出")
            else:
                print("\n操作建议:")
                print("  ✗ 无二板信号，观望")

        else:
            print("  ✗ 无法获取交易日")

    except Exception as e:
        print(f"  ✗ 执行出错: {e}")
        import traceback

        traceback.print_exc()

else:
    print("  ✗ 无候选池数据")

print("\n" + "=" * 80)
print("回测完成")
print("=" * 80)


影子策略回测 - 观察线二板（2024年）

[1] 参数配置...
  目标连板: 2板
  测试时间: 2024-01-01 至 2024-12-31

[2] 获取候选池（沪深300+中证500）...


  ✓ 候选池数量: 716

[3] 测试最近一个交易日...
  测试日期: 2024-12-31

  [3.1] 筛选二板股票...
       二板股票数量: 0只

回测结果汇总

测试日期: 2024-12-31
候选池数量: 200
二板信号: 0只

历史表现（2024实测）:
  胜率: 87.95%
  盈亏比: 21.91
  回撤: 0.60%

操作建议:
  ✗ 无二板信号，观望

回测完成
